<a href="https://colab.research.google.com/github/Rheaxu/story-converter-ai/blob/main/SaraStoryConverter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
# ==============================================================================
# CELL 1: INITIALIZATION, FILE ISOLATION, AND AI LAYOUT VISUAL ANALYSIS
# ------------------------------------------------------------------------------
# WHAT THIS CELL DOES:
# 1. Installs essential system wrappers (PyMuPDF, Edge-TTS, Google-GenAI).
# 2. Configures clean folder destinations ("storybook_images" & "storybook_audios").
# 3. Strips raw images directly out of the PDF pages.
# 4. Instructs Gemini to explicitly separate the title text block from the story
#    body text block on the first page to allow independent, sequential highlighting.
# ==============================================================================

# @title 🛠️ Step 1: Initialize Workspace & Run AI Analyzer { display-mode: "form" }
PDF_Filename = "30_MrBeeAndTed.pdf"  # @param {type:"string"}
Story_Title = "30. Mr. Bee And Ted"  # @param {type:"string"}
Folder_Safe_Name = "30_MrBeeAndTed"  # @param {type:"string"}

import os
import json
import subprocess
from google.colab import userdata

print("⚙️ Installing processing dependencies (PyMuPDF, edge-tts, google-genai)...")
subprocess.check_call(["pip", "-q", "install", "pymupdf", "edge-tts", "google-genai"])

import fitz  # PyMuPDF
from google import genai
from google.genai import types

# Establish the exact folder structures requested
project_root = Folder_Safe_Name
audio_dir = os.path.join(project_root, "storybook_audios")
images_dir = os.path.join(project_root, "storybook_images")
os.makedirs(audio_dir, exist_ok=True)
os.makedirs(images_dir, exist_ok=True)

# Verify API key placement
try:
    api_key = userdata.get('GEMINI_API_KEY')
    ai_client = genai.Client(api_key=api_key)
except Exception:
    raise ValueError("❌ Missing GEMINI_API_KEY in your Colab Secrets sidebar.")

if not os.path.exists(PDF_Filename):
    raise FileNotFoundError(f"❌ Could not find '{PDF_Filename}'. Please upload your PDF to the Colab sidebar.")

print("📖 Slicing PDF matrix and collecting raw page text...")
doc = fitz.open(PDF_Filename)
full_story_text = ""

# This loop collects text from all physical pages for Gemini analysis.
for page_num in range(len(doc)):
    page = doc[page_num]
    actual_page_idx = page_num + 1
    full_story_text += f"\n--- PDF PAGE {actual_page_idx} ---\n{page.get_text()}"

# Prompt Gemini to return the title and story text as separate structural properties
prompt = f"""
You are an advanced children's literature extraction engine. Inspect the following raw book text page by page.
Identify which pages contain vocabulary review items/word banks, and which pages are active narrative content.

CRITICAL POLICY FOR BOOK TITLES:
- Identify the very first narrative story page. Isolate the main Book Title text completely from the actual story sentence.
- For this first page ONLY, you must provide a \"title_text\" property and an \"audio_title_file\" property alongside the standard fields. Do not combine them into one text dump.

CRITICAL POLICY FOR WORD BANKS:
- Do not assume there are exactly 2 word banks.
- Check the words inside every discovered word bank. If you find multiple word banks that contain identical sets of words, consolidate them and ONLY keep one copy of that word bank in your final output map.

Return ONLY a single, perfectly valid, and unadorned JSON object. Do not include any other text, comments, or markdown formatting (e.g., ```json).
Ensure all double quotes within string values are properly escaped (e.g., \"It's \\\"fun\\\"\").
Follow this exact format:
{{
  \"word_banks\": [
     {{
       \"page_number\": 1,
       \"words\": [
          {{\"word\": \"example\", \"file\": \"wb1_0.mp3\"}},
          {{\"word\": \"hats\", \"file\": \"wb1_1.mp3\"}}
       ]
     }}
  ],
  \"story_pages\": [
     {{
       \"page_number\": 3,
       \"is_first_story_page\": true,
       \"title_text\": \"Mr. Bee and the Skunk\",
       \"audio_title_file\": \"page3_title.mp3\",
       \"display_text\": \"\\\"Hi!\\\" Mr. Bee said.\",
       \"speech_script\": \"Hi! ... Mr. Bee said.\",
       \"audio_file\": \"page3_story.mp3\",
       \"image_file\": \"page3.png\"
     }},
     {{
       \"page_number\": 4,
       \"is_first_story_page\": false,
       \"display_text\": \"\\\"Ahhhh!!!!\\\" Mr. Bee said.\",
       \"speech_script\": \"Ahhhh!!!! ... Mr. Bee said.\",
       \"audio_file\": \"page4.mp3\",
       \"image_file\": \"page4.png\"
     }}
  ]
}}

Raw Content:
{full_story_text}
"""

print("🤖 Querying Gemini to dynamically handle separate text segments...")
response = ai_client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt,
    config=types.GenerateContentConfig(response_mime_type="application/json"),
)

storybook_manifest = json.loads(response.text)
print("✔️ Text structures analyzed and sequential highlight layout compiled successfully!")

print("🖼️ Generating and cropping images for logical story pages...")

def get_physical_page_for_logical(logical_page_num):
    if logical_page_num == 1:
        return 1
    return (logical_page_num - 2) // 2 + 2

for story_page_data in storybook_manifest['story_pages']:
    logical_page_num = story_page_data['page_number']
    output_image_filename = story_page_data['image_file']
    physical_page_num = get_physical_page_for_logical(logical_page_num)

    pdf_page = doc[physical_page_num - 1]
    page_rect = pdf_page.rect
    clip_rect = None

    # VERTICAL SPLIT LOGIC:
    # Logical page 1 = Full Physical Page 1
    # Logical page 2 = Left Half of Physical Page 2
    # Logical page 3 = Right Half of Physical Page 2
    # Logical page 4 = Left Half of Physical Page 3... and so on.
    if logical_page_num == 1:
        clip_rect = page_rect
    elif logical_page_num % 2 == 0: # Even logical page numbers (2, 4, 6...) -> LEFT half
        clip_rect = fitz.Rect(page_rect.x0, page_rect.y0, page_rect.x1 / 2, page_rect.y1)
    else: # Odd logical page numbers (3, 5, 7...) -> RIGHT half
        clip_rect = fitz.Rect(page_rect.x1 / 2, page_rect.y0, page_rect.x1, page_rect.y1)

    if clip_rect:
        cropped_pix = pdf_page.get_pixmap(dpi=150, clip=clip_rect)
        cropped_pix.save(os.path.join(images_dir, output_image_filename))

print("✔️ All logical story page images generated using vertical split logic.")

⚙️ Installing processing dependencies (PyMuPDF, edge-tts, google-genai)...
📖 Slicing PDF matrix and collecting raw page text...
🤖 Querying Gemini to dynamically handle separate text segments...
✔️ Text structures analyzed and sequential highlight layout compiled successfully!
🖼️ Generating and cropping images for logical story pages...
✔️ All logical story page images generated using vertical split logic.


In [32]:
# ==============================================================================
# CELL 2: VOCAL RENDERING ENGINE
# ------------------------------------------------------------------------------
# WHAT THIS CELL DOES:
# 1. Loops through the dynamic word banks and narrative steps.
# 2. If it encounters the designated 'is_first_story_page' flag, it renders an
#    isolated voice track exclusively for the title block, then builds the separate
#    body sentence track.
# ==============================================================================

# @title 🎙️ Step 2: Render Audio Assets { display-mode: "form" }
Voice_Model = "en-US-GuyNeural"  # @param ["en-US-GuyNeural", "en-US-JennyNeural", "en-GB-SoniaNeural", "en-AU-WilliamNeural"]
Word_Bank_Speed = "-10%"  # @param ["-0%", "-5%", "-10%", "-15%", "-20%"]
Story_Reading_Speed = "-15%"  # @param ["-0%", "-5%", "-10%", "-15%", "-20%", "-25%"]

import edge_tts

async def process_vocal_pipeline():
    wb_list = storybook_manifest.get("word_banks", [])
    story_list = storybook_manifest.get("story_pages", [])

    print(f"🎙️ Rendering {len(wb_list)} vocabulary word sets...")
    for wb_idx, bank in enumerate(wb_list):
        bank_id = wb_idx + 1
        for word_idx, w_item in enumerate(bank.get("words", [])):
            filename = f"wb{bank_id}_{word_idx}.mp3"
            w_item['file'] = filename
            path = os.path.join(audio_dir, filename)
            comm = edge_tts.Communicate(w_item['word'], Voice_Model, rate=Word_Bank_Speed)
            await comm.save(path)

    print(f"🎙️ Rendering narrative pages...")
    for page in story_list:
        # Check for isolated title narrative flag
        if page.get("is_first_story_page", False) and "title_text" in page:
            title_path = os.path.join(audio_dir, page['audio_title_file'])
            title_comm = edge_tts.Communicate(page['title_text'], Voice_Model, rate=Story_Reading_Speed)
            await title_comm.save(title_path)

        # Standard page/body track rendering
        path = os.path.join(audio_dir, page['audio_file'])
        comm = edge_tts.Communicate(page['speech_script'], Voice_Model, rate=Story_Reading_Speed)
        await comm.save(path)

    print("✔️ Audio processing execution complete! All structural files rendered.")

await process_vocal_pipeline()

🎙️ Rendering 2 vocabulary word sets...
🎙️ Rendering narrative pages...
✔️ Audio processing execution complete! All structural files rendered.


In [33]:
# ==============================================================================
# CELL 3: APPLICATION COMPILER & DISTRIBUTION PACKAGER
# ------------------------------------------------------------------------------
# WHAT THIS CELL DOES:
# 1. Dynamically constructs the visual presentation layout.
# 2. Separates the text section on Page 1 into distinct Title and Story components.
# 3. Features an advanced sequential JavaScript audio queue handler that transitions
#    highlights from the Title component down to the Story text component.
# ==============================================================================

# @title 📦 Step 3: Compile Web App & Download Distribution Package { display-mode: "form" }
Story_Author = "Dr. Yvonne Crawford"  # @param {type:"string"}

import zipfile
from google.colab import files

wb_list = storybook_manifest.get("word_banks", [])
story_list = storybook_manifest.get("story_pages", [])

# Build dynamic vocabulary sheets
word_banks_html_stack = ""
for wb_idx, bank in enumerate(wb_list):
    bank_id = wb_idx + 1
    items_html = "".join([f'\n            <div class="word-item" onclick="playAudioFile(this, \'{w["file"]}\')">{w["word"]}</div>' for w in bank.get("words", [])])
    word_banks_html_stack += f"""
    <div class="page-card">
        <div class="page-number">Word Bank {bank_id}</div>
        <p style="margin:0 0 10px 0; font-weight:bold; color:#777;">Vocabulary Review:</p>
        <div class="word-bank">{items_html}</div>
    </div>"""

# Build story pages layout columns
story_pages_html_stack = ""
for page in story_list:
    if page.get("is_first_story_page", False):
        # First page layout: Split Title and Body blocks with custom structured sequential triggers
        text_panel_content = f"""
                <div style="width: 100%;">
                    <div id="first-page-title" class="story-text" style="font-size: 2rem; font-weight: bold; margin-bottom: 15px; border-bottom: 2px dashed #ccc; padding-bottom: 10px;"
                         onclick="playSequentialFirstPage('storybook_audios/{page['audio_title_file']}', 'storybook_audios/{page['audio_file']}')">
                        {page['title_text']}
                    </div>
                    <div id="first-page-body" class="story-text" onclick="playAudioFile(this, '{page['audio_file']}')">
                        {page['display_text']}
                    </div>
                </div>"""
    else:
        # Standard page layout block
        text_panel_content = f"""
                <div class="story-text" onclick="playAudioFile(this, '{page['audio_file']}')">
                    {page['display_text']}
                </div>"""

    story_pages_html_stack += f"""
    <div class="page-card">
        <div class="page-number">Page {page['page_number']}</div>
        <div class="story-layout">
            <div class="illustration-side">
                <img src="storybook_images/{page['image_file']}" alt="Page {page['page_number']}" class="story-image">
            </div>
            <div class="text-side">
                {text_panel_content}
            </div>
        </div>
    </div>"""

complete_html_payload = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{Story_Title}</title>
    <style>
        :root {{ --primary: #FFD700; --secondary: #333333; --background: #eef2f5; --card-bg: #ffffff; --text-color: #2c3e50; --interactive-hover: #eaf5ff; --interactive-active: #c6e4ff; }}
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: var(--background); color: var(--text-color); margin: 0; padding: 20px; display: flex; flex-direction: column; align-items: center; }}
        .container {{ max-width: 850px; width: 100%; }}
        header {{ text-align: center; margin-bottom: 25px; background: linear-gradient(135deg, var(--primary), #ffb700); padding: 20px; border-radius: 16px; box-shadow: 0 4px 15px rgba(0,0,0,0.08); }}
        header h1 {{ margin: 0; font-size: 2.2rem; color: var(--secondary); }}
        header p {{ margin: 5px 0 0 0; font-size: 1rem; color: #444; font-weight: 500; }}
        .audio-hint {{ display: flex; align-items: center; justify-content: center; gap: 8px; margin-bottom: 25px; font-size: 0.95rem; color: #555; font-weight: 500; }}
        .audio-hint svg {{ width: 20px; height: 20px; fill: currentColor; }}
        .page-card {{ background-color: var(--card-bg); border-radius: 16px; box-shadow: 0 6px 20px rgba(0,0,0,0.05); padding: 25px; margin-bottom: 30px; position: relative; box-sizing: border-box; }}
        .story-layout {{ display: flex; gap: 25px; align-items: center; margin-top: 10px; }}
        .illustration-side {{ flex: 0 0 55%; max-width: 55%; }}
        .story-image {{ width: 100%; height: auto; display: block; border-radius: 12px; border: 3px solid var(--secondary); box-shadow: 0 4px 10px rgba(0,0,0,0.08); background-color: #fafafa; }}
        .text-side {{ flex: 1; display: flex; align-items: center; }}
        .page-number {{ position: absolute; top: 15px; right: 20px; font-size: 0.8rem; font-weight: bold; color: #bbb; text-transform: uppercase; letter-spacing: 1px; }}
        .word-bank {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(110px, 1fr)); gap: 12px; margin-top: 10px; }}
        .word-item {{ background-color: #f8fafc; border: 2px dashed #cbd5e1; border-radius: 10px; padding: 12px 6px; text-align: center; font-size: 1.15rem; font-weight: 600; cursor: pointer; user-select: none; transition: all 0.2s; }}
        .word-item:hover {{ background-color: #f0fdf4; border-color: #4ade80; transform: scale(1.03); }}
        .story-text {{ font-size: 1.65rem; line-height: 2.4rem; font-weight: 500; color: #1a1a1a; padding: 15px; border-radius: 10px; cursor: pointer; transition: all 0.2s; width: 100%; box-sizing: border-box; border: 2px solid transparent; }}
        .story-text:hover {{ background-color: var(--interactive-hover); color: #0066cc; border-color: #bae1ff; }}
        .speaking {{ background-color: var(--interactive-active) !important; color: #004499 !important; border-color: #0066cc !important; }}
        @media(max-width: 768px) {{ .story-layout {{ flex-direction: column; }} .illustration-side {{ flex: 0 0 100%; max-width: 100%; }} }}
    </style>
</head>
<body>
<div class="container">
    <header><h1>{Story_Title}</h1><h3>by {Story_Author}</h3><p>Interactive Read-Along Experience</p></header>
    <div class="audio-hint">
        <svg viewBox="0 0 24 24"><path d="M3 9v6h4l5 5V4L7 9H3zm13.5 3c0-1.77-1.02-3.25-2.5-4.01v8.03c1.48-.76 2.5-2.24 2.5-4.02zM14 3.23v2.06c2.89.86 5 3.54 5 6.71s-2.11 5.85-5 6.71v2.06c4.01-.91 7-4.49 7-8.77s-2.99-7.86-7-8.77z"/></svg>
        <span>Click elements to play professional voice track!</span>
    </div>
    {word_banks_html_stack}
    {story_pages_html_stack}
</div>
<script>
    let currentAudio = null; let currentElement = null;

    function resetActiveStates() {{
        if (currentAudio) {{ currentAudio.pause(); currentAudio.currentTime = 0; }}
        if (currentElement) {{ currentElement.classList.remove('speaking'); }}
    }}

    function playAudioFile(element, filename) {{
        resetActiveStates();
        currentElement = element; currentElement.classList.add('speaking');

        // Dynamic directory assignment layout
        let src = filename.startsWith('storybook_audios/') ? filename : 'storybook_audios/' + filename;
        currentAudio = new Audio(src);
        currentAudio.play().catch(error => {{ console.log("Playback issue:", error); currentElement.classList.remove('speaking'); }});
        currentAudio.onended = function() {{ if (currentElement) {{ currentElement.classList.remove('speaking'); }} }};
    }}

    function playSequentialFirstPage(titleAudioSrc, bodyAudioSrc) {{
        resetActiveStates();

        let titleEl = document.getElementById('first-page-title');
        let bodyEl = document.getElementById('first-page-body');

        currentElement = titleEl;
        currentElement.classList.add('speaking');

        currentAudio = new Audio(titleAudioSrc);
        currentAudio.play().catch(e => console.log(e));

        currentAudio.onended = function() {{
            titleEl.classList.remove('speaking');

            // Introduce the requested natural breathing delay before starting body narration
            setTimeout(function() {{
                currentElement = bodyEl;
                currentElement.classList.add('speaking');
                currentAudio = new Audio(bodyAudioSrc);
                currentAudio.play().catch(e => console.log(e));
                currentAudio.onended = function() {{ bodyEl.classList.remove('speaking'); }};
            }}, 1200); // 1.2 second dramatic pause
        }};
    }}
</script>
</body>
</html>"""

with open(os.path.join(project_root, "index.html"), "w", encoding="utf-8") as f:
    f.write(complete_html_payload)

print("📦 Packaging pipeline zip module distribution payload...")
zip_filename = f"{Folder_Safe_Name}_production_package.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, file_list in os.walk(project_root):
        for file in file_list:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, os.path.dirname(project_root))
            zipf.write(file_path, arcname)

files.download(zip_filename)
print("🚀 Complete bundle dispatched directly to downloads archive!")

📦 Packaging pipeline zip module distribution payload...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🚀 Complete bundle dispatched directly to downloads archive!


In [30]:
# ==============================================================================
# CELL 4: WORKSPACE CLEANUP ENGINE
# ------------------------------------------------------------------------------
# WHAT THIS CELL DOES:
# 1. Provides a graphical form field for you to specify any target directory.
# 2. Checks if that specific folder exists in your current Colab environment.
# 3. Recursively deletes all files, images, audios, and subfolders inside it
#    to cleanly reset your workspace.
# ==============================================================================

# @title 🧹 Step 4: Clear Specified Project Directory { display-mode: "form" }
target_directory = "30_MrBeeAndTed"  # @param {type:"string"}

import shutil
import os

# Clean up any trailing slashes user might type by accident
target_directory = target_directory.strip().rstrip('/')

if target_directory and os.path.exists(target_directory):
    print(f"⚠️ Initializing cleanup sequence for directory: '{target_directory}'...")

    try:
        # Recursively deletes the entire directory tree
        shutil.rmtree(target_directory)
        print(f"🗑️ Clean sweep complete! '{target_directory}' and all of its contents have been deleted.")
    except Exception as e:
        print(f"❌ An error occurred while trying to delete the folder: {e}")

elif not target_directory:
    print("❌ Error: Please specify a valid folder name in the target_directory box.")
else:
    print(f"ℹ️ The folder '{target_directory}' does not exist or has already been cleared.")

⚠️ Initializing cleanup sequence for directory: '30_MrBeeAndTed'...
🗑️ Clean sweep complete! '30_MrBeeAndTed' and all of its contents have been deleted.
